# 🧠 Stage 2: Baseline Model Evaluation & Collapse Diagnosis
**Project:** Reduction Ladder for Code & Multi-Arm Mitigation  
**Organization:** Orange Innovation Labs — AI Research Division  
**Authors:** Omar Abdelhamid, Nour Walid  
**Supervisor:** Dr. Ghada  

---

### 🎯 Objectives of this Notebook:
1. Load the un-tuned student model **`Qwen/Qwen2.5-Coder-1.5B-Instruct`** in **4-bit NF4 precision** (fits comfortably in 4 GB VRAM on our RTX 3070).
2. Execute deterministic **Pass@1** (greedy decoding, $T=0.0$) and **Pass@5** (nucleus sampling, $T=0.8$) across all six ladder levels (**L0 to L5**).
3. Run automated **Error Taxonomy Diagnosis** (categorizing failures into `on_path`, `off_path`, `wrong_template`, `syntax_error`, or `runtime_error`).
4. Determine the baseline **Collapse Point** ($\ell^*$) and plot the un-tuned degradation curve.

## 1. Environment Setup & Clean Architecture Imports

In [ ]:
import os
import sys
import torch
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
sys.path.insert(0, os.path.abspath(".."))

from src.services.data_service import DataService
from src.services.evaluation_service import EvaluationService
from src.services.analysis_service import AnalysisService
from src.infrastructure.hf_loader import HuggingFaceBenchmarkLoader
from src.infrastructure.model_loader import QuantizedModelRunner
from src.infrastructure.sandbox import MultiprocessSandbox
from src.infrastructure.classifier import RuleBasedErrorClassifier

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Initial VRAM Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB")

print("\n✅ Clean Architecture modules imported successfully!")

## 2. Load the Reduction Ladder Benchmarks (L0 to L5)

In [ ]:
data_service = DataService(loader=HuggingFaceBenchmarkLoader(cache_dir="../data/ladder"))
ladder_data = data_service.prepare_all_benchmarks(force_download=False)

for lvl, tasks in ladder_data.items():
    print(f"Level {lvl} ({tasks[0].benchmark}): {len(tasks)} tasks loaded")

## 3. Initialize Quantized Model Runner (4-bit NF4)
We load `Qwen/Qwen2.5-Coder-1.5B-Instruct` using bitsandbytes 4-bit Double Quantization to stay well within our 8 GB VRAM budget.

In [ ]:
model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
model_runner = QuantizedModelRunner(model_name_or_path=model_name)

eval_service = EvaluationService(
    model_runner=model_runner,
    executor=MultiprocessSandbox(default_timeout=5.0),
    classifier=RuleBasedErrorClassifier()
)

if torch.cuda.is_available():
    print(f"\n🔥 Post-load VRAM Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB")

## 4. Run Full Ladder Evaluation (L0 to L5)
We evaluate Pass@1 (Greedy) and diagnose error categories for every task across all 6 levels.

In [ ]:
os.makedirs("../results/baseline", exist_ok=True)
output_report_file = "../results/baseline/baseline_evaluation_report.json"

baseline_suite_report = eval_service.evaluate_suite(
    ladder_data=ladder_data,
    model_name="M1: Baseline (Qwen2.5-Coder-1.5B-Instruct)",
    evaluate_pass5=False,  # Set to True for full exploration sampling
    output_file=output_report_file
)

## 5. Metrics Summary & Collapse Point Determination

In [ ]:
results_dict = {
    "M1: Baseline": {
        "ladder_auc": baseline_suite_report.ladder_auc,
        "collapse_point": baseline_suite_report.collapse_point,
        "consistency_delta": baseline_suite_report.consistency_delta,
        "level_reports": {
            k: {"pass_at_1": v.pass_at_1, "error_breakdown": v.error_breakdown}
            for k, v in baseline_suite_report.level_reports.items()
        }
    }
}

summary_table = AnalysisService.compute_summary_table(results_dict)
summary_table

## 6. Plot the Baseline Degradation Curve

In [ ]:
AnalysisService.plot_ladder_curves(
    suite_reports=results_dict,
    output_filepath="../results/baseline/baseline_degradation_curve.png"
)

# Display plot inside notebook
from IPython.display import Image
Image("../results/baseline/baseline_degradation_curve.png")

## 7. Error Taxonomy Breakdown Across Levels
Let's inspect what types of errors caused the collapse at higher ladder levels (e.g. wrong template vs on-path vs off-path).

In [ ]:
error_records = []
for lvl, rep in baseline_suite_report.level_reports.items():
    breakdown = rep.error_breakdown.copy()
    breakdown["Level"] = lvl
    error_records.append(breakdown)

error_df = pd.DataFrame(error_records).set_index("Level").fillna(0)
print("📊 Error Counts by Category:")
display(error_df)

# Plot Stacked Error Distribution
plt.figure(figsize=(10, 5), dpi=150)
error_df.drop(columns=["pass"], errors="ignore").plot(
    kind="bar",
    stacked=True,
    colormap="Set2",
    figsize=(10, 5)
)
plt.title("M1 Baseline: Error Taxonomy Breakdown Across Ladder Levels")
plt.xlabel("Ladder Level")
plt.ylabel("Number of Tasks")
plt.legend(title="Error Category", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("../results/baseline/baseline_error_taxonomy.png")
plt.show()

## 8. Inspect Concrete Failure Examples (Template Recalls)
Let's print specific tasks from L2/L3 where the model failed due to `wrong_template` or `off_path` errors.

In [ ]:
for lvl in ["L1", "L2", "L3"]:
    if lvl in baseline_suite_report.level_reports:
        rep = baseline_suite_report.level_reports[lvl]
        failures = [t for t in rep.task_results if not t["p1_passed"]]
        if failures:
            sample = failures[0]
            print(f"\n{'='*30} Sample Failure in {lvl} (Task ID: {sample['task_id']}) {'='*30}")
            print(f"Diagnosis: {sample['p1_error_category']}")
            print(f"Error Message: {sample['p1_error_message']}")
            print("Generated Code Snippet:")
            print(sample["p1_code"][:400] + "...")